# Running GPT-Type Models Locally with Ollama — Completed Lab

**Sources:** `Run_Ollama.ipynb` and the Ollama activity in `Transformer.pdf`

The notebook installs/uses Ollama when available, runs the original LangChain pattern, issues several prompts, checks responses, and provides completed reference answers and a GPT/Claude comparison. Model calls require a local Ollama service and the `mistral` model; the written solutions and deterministic code tests run without them.

## 1. Installation and model setup

**Local computer:** install Ollama from <https://ollama.com/download>, open a terminal, and run:

```bash
ollama serve       # omit if the desktop service is already running
ollama pull mistral
ollama list
```

**Google Colab (Linux):** run the shell commands below in a code cell. Colab's runtime is temporary, so the model must be downloaded again after a reset.

```bash
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &
!ollama pull mistral
!pip install -q -U langchain-ollama langchain-core
```

Do not use `%xterm` for the automated solution: starting the service and pulling the model explicitly is reproducible and works in both local Jupyter and Colab.

In [1]:
import json
import shutil
import subprocess
from pathlib import Path

OLLAMA_AVAILABLE = shutil.which("ollama") is not None
print("Ollama executable found:", OLLAMA_AVAILABLE)
if OLLAMA_AVAILABLE:
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=30)
    print(result.stdout or result.stderr)
else:
    print("Run the installation commands above, then rerun this cell.")

Ollama executable found: False
Run the installation commands above, then rerun this cell.


## 2. Run prompts through LangChain and Ollama

The prompt template explicitly requests step-by-step reasoning. The response checks test content rather than requiring exact wording, because probabilistic model outputs vary between runs and versions.

In [2]:
questions = [
    "Who is Sir Isaac Newton?",
    "Explain self-attention in two sentences for a beginner.",
    "Write a Python function that returns the sum of the even numbers in a list.",
]

responses = {}
if not OLLAMA_AVAILABLE:
    print("Model calls skipped: install Ollama and pull mistral using Section 1.")
else:
    try:
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_ollama.llms import OllamaLLM
    except ImportError:
        print("Install dependencies with: pip install -U langchain-ollama langchain-core")
    else:
        template = """Question: {question}

Answer: Let's think step by step."""
        prompt = ChatPromptTemplate.from_template(template)
        model = OllamaLLM(model="mistral")
        chain = prompt | model
        for question in questions:
            answer = chain.invoke({"question": question})
            responses[question] = answer
            print("Q:", question)
            print("A:", answer)
            print()
        Path("ollama_responses.json").write_text(
            json.dumps(responses, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        print("Saved model outputs to ollama_responses.json")

Model calls skipped: install Ollama and pull mistral using Section 1.


## 3. Completed reference answers

**Who is Sir Isaac Newton?** Sir Isaac Newton (1642–1727) was an English mathematician, physicist, astronomer, and author. He formulated the three laws of motion and the law of universal gravitation, made major contributions to optics, and independently developed calculus. His *Philosophiæ Naturalis Principia Mathematica* (1687) was foundational for classical mechanics.

**Self-attention in two sentences:** Self-attention lets each token use information from other tokens in the same sequence by calculating how relevant they are. It then combines their value vectors using those relevance weights, allowing the model to use context rather than treating each token independently.

**Correct Python function:**

```python
def sum_even(numbers):
    return sum(number for number in numbers if number % 2 == 0)
```

These are reference answers for checking an Ollama response, not fabricated Ollama output.

In [3]:
def sum_even(numbers):
    return sum(number for number in numbers if number % 2 == 0)

assert sum_even([1, 2, 3, 4, 5, 6]) == 12
assert sum_even([]) == 0
assert sum_even([-4, -3, 0, 8]) == 4
print("Reference coding solution: all tests passed.")

# Automatic content checks run only when real Ollama responses exist.
if responses:
    newton = responses[questions[0]].lower()
    assert "newton" in newton and ("gravity" in newton or "gravitation" in newton)
    attention = responses[questions[1]].lower()
    assert "attention" in attention and ("token" in attention or "word" in attention)
    coding = responses[questions[2]]
    assert "def" in coding and "% 2" in coding.replace("%2", "% 2")
    print("All three Ollama responses passed the content checks.")

Reference coding solution: all tests passed.


## 4. Comparison with GPT/Claude

Use the same three prompts and compare **content**, not just fluency.

| Criterion | Local Ollama + Mistral | Hosted GPT/Claude |
|---|---|---|
| Factual answer | Should include Newton's laws/gravitation and may be shorter or less nuanced | Usually detailed and well structured; still requires factual checking |
| Beginner explanation | Check that relevance weighting and contextual mixing are both present | Often provides clearer analogies and more polished explanations |
| Code | Check the modulo condition, accumulation, empty input, and negative even numbers | Often adds explanation and tests; generated code still must be executed |
| Privacy | Prompts remain on the local machine | Prompts are sent to a hosted service subject to its data controls |
| Setup/resources | Requires installation, model download, RAM/VRAM, and local inference time | Minimal local setup; requires network access and an account/API |
| Cost/customisation | No per-token fee after setup; easy to switch open models | Usage may be billed; model choice and deployment are managed by provider |
| Reproducibility | Record model tag, Ollama version, prompt, and decoding settings | Record provider, model version, prompt, and settings |

**Conclusion:** local Ollama is preferable when privacy, offline operation, experimentation, and model control matter most. Hosted GPT/Claude is often preferable when convenience and stronger response quality matter more than local control. Neither should be trusted without the factual and executable checks above.

## 5. Lab checklist

- Ollama installation and `mistral` setup: documented for local and Colab.
- `ollama list` and LangChain `OllamaLLM`: implemented.
- Original Newton prompt plus explanation and coding prompts: implemented with response saving.
- Reference answers and coding tests: completed.
- GPT/Claude comparison: completed across correctness, privacy, resources, cost, and reproducibility.